# Bài thực hành dữ liệu đặt phòng khách sạn — Notebook khung

**Phiên bản 2.0 · Dành cho sinh viên · Chưa có lời giải**

| Thông tin nhóm | Điền vào đây |
|---|---|
| Mã nhóm | NhomXX |
| Thành viên 1 (MSSV) — phần phụ trách | … |
| Thành viên 2 (MSSV) — phần phụ trách | … |
| Thành viên 3 (MSSV) — phần phụ trách | … |
| Ngày chạy lại toàn bộ lần cuối | … |

**Trước khi nộp:** đổi tên tệp thành `hotel_analysis.ipynb`.

## Cách dùng notebook này

Notebook đi theo đúng 5 giai đoạn của đề bài: **A Khảo sát → B Làm sạch → C Bảng sạch & kiểm tra → D Phân tích 10 câu → E Xuất & nộp**.

| Ký hiệu | Ý nghĩa | Bạn cần làm gì |
|---|---|---|
| 📋 Thẻ nhiệm vụ | Bảng đầu mỗi mục: mục tiêu, đầu vào/ra, các bước, bẫy | Đọc trước khi viết mã |
| `# TODO` | Chỗ nhóm phải viết mã | Thay bằng mã của nhóm; xóa chữ TODO khi xong |
| `= None` | Biến đầu ra nhóm phải tạo | Gán DataFrame thật, **giữ đúng tên biến** |
| ✅ / ❌ | Kết quả ô tự kiểm tra | ❌ nghĩa là chưa xong hoặc chưa đúng quy ước — đọc gợi ý in kèm |
| ✍️ | Ô văn bản nhóm phải viết | Viết bằng số liệu thật, có mẫu số |

**Ba quy tắc vàng**

1. **Không sửa tay** `hotel_bookings_dirty.csv`, không thay bằng bộ dữ liệu gốc sạch.
2. **Không xóa các ô kiểm tra** (✅/❌). Chúng giúp bạn và giảng viên thấy tiến độ.
3. Cuối mỗi buổi: **Runtime/Kernel → Restart & Run all**. Notebook phải chạy được từ đầu đến cuối.

Notebook vẫn chạy hết khi còn dang dở: các ô kiểm tra chỉ in ❌, không làm dừng chương trình.

## 0 · Thiết lập

**Colab:** File → Upload notebook; mở bảng 📁 Files, tải `hotel_bookings_dirty.csv` lên. Có thể tải thêm `kiem_tra_nop_bai.py` để tự kiểm tra ở cuối.
**Máy cá nhân:** giữ cấu trúc thư mục của gói bài tập (notebook ở thư mục gốc, dữ liệu trong `data/`).

Ô 0.1 chứa công cụ hỗ trợ dùng chung. **Không cần sửa**, chỉ cần chạy.

In [ ]:
# 0.1 CÔNG CỤ HỖ TRỢ — KHÔNG CẦN SỬA
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 40)

# ---- Đường dẫn: tự tìm dữ liệu ở data/ hoặc cùng thư mục notebook (Colab) ----
CANDIDATES = [Path('data/hotel_bookings_dirty.csv'), Path('hotel_bookings_dirty.csv')]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), CANDIDATES[0])
OUTPUT_DIR = Path('outputs')
for sub in ['', 'figures', 'tables']:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

# ---- Quy ước của đề (Mục 5) ----
SNAPSHOT_DATE = pd.Timestamp('2017-09-14')
MISSING_TOKENS = ['NULL', 'N/A', 'NA', '?', 'unknown', '-']        # so sánh không phân biệt hoa thường
HOTEL_COLORS = {'City Hotel': '#2F4858', 'Resort Hotel': '#E07A1F'}  # dùng cho MỌI hình có hotel
STATUS_COLORS = {'Check-Out': '#5B8E7D', 'Canceled': '#B81D24', 'No-Show': '#F2A541'}
FIGSIZE = (9, 5)

# ===================== BỘ KIỂM TRA ĐẦU RA (dùng chung notebook + tools) =====================
# Bộ kiểm tra chỉ xác nhận CẤU TRÚC, KHÓA, MIỀN GIÁ TRỊ và TÍNH NHẤT QUÁN theo quy ước trong đề.
# Nó KHÔNG chứa đáp án và KHÔNG chứng minh kết quả phân tích của nhóm là đúng.
import pandas as pd

HOTELS = ['City Hotel', 'Resort Hotel']
MEALS = ['BB', 'HB', 'FB', 'SC']
SEGMENTS = ['Online TA', 'Offline TA/TO', 'Groups', 'Direct', 'Corporate', 'Complementary', 'Aviation']
CHANNELS = ['TA/TO', 'Direct', 'Corporate', 'GDS']
DEPOSITS = ['No Deposit', 'Non Refund', 'Refundable']
CUSTOMERS = ['Transient', 'Transient-Party', 'Contract', 'Group']
STATUSES = ['Check-Out', 'Canceled', 'No-Show']
ROOM_TYPES = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'P']
LEAD_GROUPS = ['0-7', '8-30', '31-90', '91-180', '181+']

ARRIVAL_MIN, ARRIVAL_MAX = pd.Timestamp('2015-07-01'), pd.Timestamp('2017-08-31')
STATUS_MIN, STATUS_MAX = pd.Timestamp('2014-10-01'), pd.Timestamp('2017-09-14')
LEAD_MIN, LEAD_MAX = 0, 800
WEEKEND_MAX, WEEK_MAX = 20, 50
GUEST_MAX = 10
ADR_MIN, ADR_MAX = 0, 1000
REQ_MAX = 5
N_RAW_ROWS = 10325

CLEAN_REQUIRED = ['booking_id', 'hotel', 'is_canceled', 'lead_time', 'arrival_date', 'stays_in_weekend_nights',
                  'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment',
                  'distribution_channel', 'is_repeated_guest', 'reserved_room_type', 'assigned_room_type',
                  'deposit_type', 'customer_type', 'adr', 'total_of_special_requests', 'reservation_status',
                  'reservation_status_date',
                  'arrival_year', 'arrival_month', 'total_nights', 'total_guests', 'lead_time_group',
                  'room_changed_flag', 'revenue_est']
SUMMARY_COLUMNS = ['hotel', 'arrival_year', 'arrival_month', 'n_bookings', 'n_canceled',
                   'cancel_rate_pct', 'room_nights']
LOG_COLUMNS = ['step', 'field', 'issue', 'detection_rule', 'action', 'affected_rows',
               'changed_cells', 'removed_rows', 'example_row_ids', 'reason', 'check_after']
LINEAGE_COLUMNS = ['row_id', 'booking_id', 'status']
LINEAGE_STATUS = ['kept', 'merged', 'dropped']
QUALITY_COLUMNS = ['column', 'stage', 'denominator_rows', 'missing_count', 'invalid_count', 'missing_pct']

CORE_FILES = ['bookings_clean.csv', 'monthly_summary.csv', 'cleaning_log.csv', 'row_lineage.csv',
              'quality_report.csv']
FIGURE_FILES = ['fig01_hotel_cancel.png', 'fig02_monthly_arrivals_2016.png', 'fig03_leadtime_cancel.png',
                'fig04_top10_country.png', 'fig05_segment_cancel.png', 'fig06_deposit_cancel.png',
                'fig07_adr_hist.png', 'fig08_stay_nights.png']
TABLE_FILES = ['a_missing_survey.csv', 'a_issue_catalog.csv', 'q01_before_after.csv',
               'q02_hotel_cancel.csv', 'q03_monthly_arrivals_2016.csv', 'q04_leadtime_stats.csv',
               'q04_leadtime_cancel.csv', 'q05_top10_country.csv', 'q06_segment_cancel.csv',
               'q07_deposit_cancel.csv', 'q08_adr_stats_by_hotel.csv', 'q09_stay_nights.csv',
               'q10_room_change_by_hotel.csv', 'q10_room_change_by_reserved_type.csv']


def _blank(s):
    """True khi ô thiếu: NaN/None/NA hoặc chuỗi rỗng sau khi bỏ khoảng trắng."""
    return s.isna() | s.astype('string').str.strip().fillna('').eq('')


def _num(s):
    return pd.to_numeric(s.where(~_blank(s)), errors='coerce')


def _int_ok(s, lo, hi):
    """Ô có dữ liệu phải là số nguyên trong [lo, hi]. Trả về (đạt?, số ô sai)."""
    present = ~_blank(s)
    v = _num(s)
    bad = present & (v.isna() | (v % 1 != 0) | (v < lo) | (v > hi))
    return int(bad.sum()) == 0, int(bad.sum())


def _bool(s):
    m = {'true': True, 'false': False, '1': True, '0': False, '1.0': True, '0.0': False}
    return s.astype('string').str.strip().str.lower().map(m)


def _date(s):
    txt = s.astype('string').str.strip()
    return pd.to_datetime(txt.where(~_blank(s)), format='%Y-%m-%d', errors='coerce')


def _cat_ok(s, allowed, allow_missing=True):
    present = ~_blank(s)
    bad = sorted(set(s[present].astype(str)) - set(allowed))
    miss = int((~present).sum())
    ok = not bad and (allow_missing or miss == 0)
    return ok, ('Nhãn lạ: ' + ', '.join(bad[:6]) if bad else '') + ('' if allow_missing or miss == 0 else f' {miss} ô thiếu')


def validate_outputs(clean, monthly, row_lineage, cleaning_log, quality_report, raw_row_ids=None):
    """Chạy toàn bộ kiểm tra. Trả về DataFrame: nhom, kiem_tra, ket_qua, chi_tiet."""
    res = []

    def add(group, name, ok, detail=''):
        res.append({'nhom': group, 'kiem_tra': name,
                    'ket_qua': 'ĐẠT' if ok else 'CHƯA ĐẠT', 'chi_tiet': detail})

    def safe(group, name, fn):
        try:
            ok, detail = fn()
        except Exception as e:  # thiếu cột, sai kiểu... -> báo lỗi thân thiện
            ok, detail = False, f'Lỗi khi kiểm tra: {type(e).__name__}: {e}'
        add(group, name, ok, detail)

    frames = {'bookings_clean': clean, 'monthly_summary': monthly, 'row_lineage': row_lineage,
              'cleaning_log': cleaning_log, 'quality_report': quality_report}
    for k, v in frames.items():
        add('0 Tồn tại', f'{k} đã được tạo', v is not None and len(v) > 0,
            '' if v is not None else 'Biến/tệp còn trống (None).')

    # ---------------- Bảng chính: khóa và nhãn ----------------
    if clean is not None:
        c = clean
        G = '1 Khóa & nhãn'
        safe(G, 'Đủ các cột bắt buộc',
             lambda: (not (miss := [x for x in CLEAN_REQUIRED if x not in c.columns]),
                      'Thiếu: ' + ', '.join(miss) if miss else f'{len(c.columns)} cột'))
        safe(G, 'booking_id không thiếu, dạng BK + 6 chữ số',
             lambda: ((n := int((_blank(c['booking_id']) | ~c['booking_id'].astype('string').str.fullmatch(r'BK\d{6}').fillna(False)).sum())) == 0,
                      f'{n} ô thiếu hoặc sai dạng'))
        safe(G, 'booking_id không trùng (1 dòng / đặt phòng)',
             lambda: ((n := int(c['booking_id'].duplicated().sum())) == 0, f'{n} booking_id lặp'))
        safe(G, 'hotel chỉ gồm City Hotel, Resort Hotel (không thiếu)', lambda: _cat_ok(c['hotel'], HOTELS, False))
        for col, allowed in [('meal', MEALS), ('market_segment', SEGMENTS), ('distribution_channel', CHANNELS),
                             ('deposit_type', DEPOSITS), ('customer_type', CUSTOMERS), ('reservation_status', STATUSES),
                             ('reserved_room_type', ROOM_TYPES), ('assigned_room_type', ROOM_TYPES)]:
            safe(G, f'{col} thuộc tập nhãn hợp lệ hoặc thiếu', lambda col=col, allowed=allowed: _cat_ok(c[col], allowed))
        safe(G, 'country là mã 2–3 chữ in hoa hoặc thiếu',
             lambda: ((n := int((~_blank(c['country']) & ~c['country'].astype('string').str.fullmatch(r'[A-Z]{2,3}').fillna(False)).sum())) == 0,
                      f'{n} ô sai dạng'))
        safe(G, 'is_canceled, is_repeated_guest chỉ là 0/1 hoặc thiếu',
             lambda: ((a := _int_ok(c['is_canceled'], 0, 1))[0] and (b := _int_ok(c['is_repeated_guest'], 0, 1))[0],
                      f'is_canceled sai: {a[1]}; is_repeated_guest sai: {b[1]}'))

        # ---------------- Số đo và ngày ----------------
        G = '2 Số & ngày'
        for col, lo, hi in [('lead_time', LEAD_MIN, LEAD_MAX), ('stays_in_weekend_nights', 0, WEEKEND_MAX),
                            ('stays_in_week_nights', 0, WEEK_MAX), ('adults', 0, GUEST_MAX), ('children', 0, GUEST_MAX),
                            ('babies', 0, GUEST_MAX), ('total_of_special_requests', 0, REQ_MAX)]:
            safe(G, f'{col} nguyên trong {lo}–{hi}', lambda col=col, lo=lo, hi=hi: ((r := _int_ok(c[col], lo, hi))[0], f'{r[1]} ô sai'))

        def _adr():
            present = ~_blank(c['adr'])
            v = _num(c['adr'])
            bad = present & (v.isna() | (v < ADR_MIN) | (v > ADR_MAX))
            return int(bad.sum()) == 0, f'{int(bad.sum())} ô không phải số trong 0–1000'
        safe(G, 'adr là số (EUR) trong 0–1000 hoặc thiếu', _adr)

        def _dates(col, lo, hi):
            s = c[col]
            present = ~_blank(s)
            fmt_bad = present & ~s.astype('string').str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}').fillna(False)
            d = _date(s)
            rng_bad = present & (d.isna() | (d < lo) | (d > hi))
            n = int((fmt_bad | rng_bad).sum())
            return n == 0, f'{n} ô sai định dạng YYYY-MM-DD hoặc ngoài {lo:%Y-%m-%d}..{hi:%Y-%m-%d}'
        safe(G, 'arrival_date dạng YYYY-MM-DD, trong 2015-07-01..2017-08-31',
             lambda: _dates('arrival_date', ARRIVAL_MIN, ARRIVAL_MAX))
        safe(G, 'reservation_status_date dạng YYYY-MM-DD, trong 2014-10-01..2017-09-14',
             lambda: _dates('reservation_status_date', STATUS_MIN, STATUS_MAX))

        def _cons():
            st = c['reservation_status'].astype('string')
            ic = _num(c['is_canceled'])
            exp = st.map({'Check-Out': 0, 'Canceled': 1, 'No-Show': 1})
            bad = exp.notna() & (ic.astype('Float64') != exp.astype('Float64')).fillna(True)
            return int(bad.sum()) == 0, f'{int(bad.sum())} dòng is_canceled mâu thuẫn reservation_status'
        safe(G, 'is_canceled nhất quán với reservation_status', _cons)

        # ---------------- Cột dẫn xuất ----------------
        G = '3 Cột dẫn xuất'

        def _ym():
            d = _date(c['arrival_date'])
            bad = (d.dt.year.astype('Float64').fillna(-1) != _num(c['arrival_year']).fillna(-1)) | \
                  (d.dt.month.astype('Float64').fillna(-1) != _num(c['arrival_month']).fillna(-1))
            return int(bad.sum()) == 0, f'{int(bad.sum())} dòng không khớp arrival_date'
        safe(G, 'arrival_year, arrival_month lấy từ arrival_date', _ym)

        def _sum(target, parts):
            exp = sum(_num(c[p]) for p in parts)
            bad = exp.fillna(-999) != _num(c[target]).fillna(-999)
            return int(bad.sum()) == 0, f'{int(bad.sum())} dòng sai (thiếu một thành phần → kết quả thiếu)'
        safe(G, 'total_nights = weekend + week nights', lambda: _sum('total_nights', ['stays_in_weekend_nights', 'stays_in_week_nights']))
        safe(G, 'total_guests = adults + children + babies', lambda: _sum('total_guests', ['adults', 'children', 'babies']))

        def _lead():
            lt = _num(c['lead_time'])
            exp = pd.cut(lt, bins=[-0.5, 7.5, 30.5, 90.5, 180.5, float('inf')], labels=LEAD_GROUPS).astype('string')
            got = c['lead_time_group'].astype('string').where(~_blank(c['lead_time_group']))
            bad = exp.fillna('∅') != got.fillna('∅')
            return int(bad.sum()) == 0, f'{int(bad.sum())} dòng sai nhóm (nhãn: {", ".join(LEAD_GROUPS)})'
        safe(G, 'lead_time_group đúng 5 nhóm quy ước', _lead)

        def _room():
            r, a = c['reserved_room_type'], c['assigned_room_type']
            exp = (~_blank(r) & ~_blank(a) & (r.astype('string') != a.astype('string'))).fillna(False)
            got = _bool(c['room_changed_flag'])
            bad = got.isna() | (got.astype('boolean') != exp.astype('boolean'))
            return int(bad.sum()) == 0, f'{int(bad.sum())} dòng sai hoặc thiếu cờ (chỉ dùng True/False)'
        safe(G, 'room_changed_flag đúng quy tắc', _room)

        def _rev():
            adr, n, ic = _num(c['adr']), _num(c['total_nights']), _num(c['is_canceled'])
            exp = (adr * n).where(ic == 0).round(2)
            got = _num(c['revenue_est']).round(2)
            same = (exp.isna() & got.isna()) | ((exp - got).abs() <= 0.011).fillna(False)
            bad = ~same
            return int(bad.sum()) == 0, f'{int(bad.sum())} dòng sai (chỉ tính khi is_canceled = 0 và adr, total_nights hợp lệ)'
        safe(G, 'revenue_est = adr × total_nights (chỉ đặt phòng không hủy)', _rev)

    # ---------------- Bảng tổng hợp theo tháng ----------------
    if monthly is not None:
        m = monthly
        G = '4 Bảng tháng'
        safe(G, 'monthly_summary đúng 7 cột',
             lambda: (list(m.columns) == SUMMARY_COLUMNS, f'Cột hiện có: {list(m.columns)}'))
        safe(G, 'Không trùng khóa (hotel, arrival_year, arrival_month)',
             lambda: ((n := int(m.duplicated(['hotel', 'arrival_year', 'arrival_month']).sum())) == 0, f'{n} khóa lặp'))
        if clean is not None:
            def _recompute():
                c = clean.copy()
                c['_y'], c['_m'] = _num(c['arrival_year']), _num(c['arrival_month'])
                c['_ic'], c['_n'] = _num(c['is_canceled']), _num(c['total_nights'])
                c = c[c['_y'].notna()]
                g = c.groupby([c['hotel'].astype(str), '_y', '_m'])
                exp = pd.DataFrame({'n_bookings': g.size(), 'n_canceled': g['_ic'].apply(lambda s: (s == 1).sum()),
                                    'known': g['_ic'].apply(lambda s: s.notna().sum()),
                                    'room_nights': g.apply(lambda x: x.loc[x['_ic'] == 0, '_n'].sum())}).reset_index()
                exp.columns = ['hotel', 'arrival_year', 'arrival_month', 'n_bookings', 'n_canceled', 'known', 'room_nights']
                exp['cancel_rate_pct'] = (exp['n_canceled'] / exp['known'] * 100).round(2)
                got = m.copy()
                for col in ['arrival_year', 'arrival_month', 'n_bookings', 'n_canceled', 'cancel_rate_pct', 'room_nights']:
                    got[col] = _num(got[col])
                got['hotel'] = got['hotel'].astype(str)
                j = exp.merge(got, on=['hotel', 'arrival_year', 'arrival_month'], how='outer', suffixes=('_e', '_g'), indicator=True)
                miss = int((j['_merge'] != 'both').sum())
                b = j[j['_merge'] == 'both']
                bad = int(((b['n_bookings_e'] != b['n_bookings_g']) | (b['n_canceled_e'] != b['n_canceled_g']) |
                           ~(((b['cancel_rate_pct_e'] - b['cancel_rate_pct_g']).abs() <= 0.011) |
                             (b['cancel_rate_pct_e'].isna() & b['cancel_rate_pct_g'].isna())) |
                           ((b['room_nights_e'] - b['room_nights_g']).abs() > 0.5)).sum())
                return miss == 0 and bad == 0, f'{miss} tháng thiếu/thừa; {bad} tháng lệch số liệu so với bảng chính'
            safe(G, 'Số liệu khớp khi tính lại từ bookings_clean', _recompute)

    # ---------------- Truy vết ----------------
    if row_lineage is not None:
        rl = row_lineage
        G = '5 Truy vết'
        safe(G, 'row_lineage có cột row_id, booking_id, status',
             lambda: (all(x in rl.columns for x in LINEAGE_COLUMNS), f'Cột hiện có: {list(rl.columns)}'))
        safe(G, 'Mỗi row_id xuất hiện đúng 1 lần',
             lambda: ((n := int(rl['row_id'].duplicated().sum())) == 0, f'{n} row_id lặp'))
        if raw_row_ids is not None:
            safe(G, f'Phủ đủ {N_RAW_ROWS} row_id gốc',
                 lambda: ((mm := len(set(raw_row_ids) - set(rl['row_id']))) == 0 and
                          (x := len(set(rl['row_id']) - set(raw_row_ids))) == 0,
                          f'Thiếu {mm} row_id gốc; thừa {x} row_id lạ'))
        safe(G, 'status thuộc kept / merged / dropped',
             lambda: (not (bad := sorted(set(rl['status'].astype(str)) - set(LINEAGE_STATUS))),
                      'Giá trị lạ: ' + ', '.join(bad[:5]) if bad else ''))
        if clean is not None:
            safe(G, 'Mỗi booking_id trong bảng chính có đúng 1 dòng kept',
                 lambda: ((k := rl.loc[rl['status'] == 'kept', 'booking_id']).is_unique and
                          set(k) == set(clean['booking_id']),
                          f'{int(k.duplicated().sum())} booking_id có >1 kept; '
                          f'{len(set(clean["booking_id"]) - set(k))} booking_id không có kept'))
            safe(G, 'Dòng kept/merged trỏ tới booking_id có trong bảng chính',
                 lambda: ((n := int((rl['status'].isin(['kept', 'merged']) &
                                     ~rl['booking_id'].isin(clean['booking_id'])).sum())) == 0, f'{n} dòng trỏ sai'))

    # ---------------- Nhật ký & chất lượng ----------------
    G = '6 Nhật ký'
    if cleaning_log is not None:
        lg = cleaning_log
        safe(G, 'cleaning_log đúng 11 cột theo mẫu',
             lambda: (list(lg.columns) == LOG_COLUMNS, f'Cột hiện có: {list(lg.columns)}'))
        safe(G, 'Có ít nhất 10 quy tắc được ghi', lambda: (len(lg) >= 10, f'{len(lg)} dòng'))
        safe(G, 'affected_rows, changed_cells, removed_rows là số nguyên ≥ 0',
             lambda: (all(_int_ok(lg[x], 0, 10**7)[0] and (~_blank(lg[x])).all()
                          for x in ['affected_rows', 'changed_cells', 'removed_rows']), ''))
        safe(G, 'Mỗi quy tắc có lý do (reason) và kiểm tra sau (check_after)',
             lambda: ((n := int((_blank(lg['reason']) | _blank(lg['check_after'])).sum())) == 0, f'{n} dòng thiếu'))
    if quality_report is not None:
        q = quality_report
        safe(G, 'quality_report có các cột bắt buộc',
             lambda: (not (mm := [x for x in QUALITY_COLUMNS if x not in q.columns]), 'Thiếu: ' + ', '.join(mm) if mm else ''))
        safe(G, 'quality_report có cả before và after',
             lambda: ({'before', 'after'} <= set(q['stage'].astype(str)), f'stage: {sorted(set(q["stage"].astype(str)))}'))

    return pd.DataFrame(res)


def print_report(report):
    n_ok = int((report['ket_qua'] == 'ĐẠT').sum())
    for _, r in report.iterrows():
        mark = '✅' if r['ket_qua'] == 'ĐẠT' else '❌'
        extra = f'  — {r["chi_tiet"]}' if r['chi_tiet'] and r['ket_qua'] != 'ĐẠT' else ''
        print(f'{mark} [{r["nhom"]}] {r["kiem_tra"]}{extra}')
    print(f'\nTỔNG KẾT: {n_ok}/{len(report)} kiểm tra đạt.')
    return n_ok == len(report)
# =============================================================================================


# ---- Hàm tiện ích ----
cleaning_log_rows = []   # mỗi phần tử là 1 dict = 1 dòng nhật ký

def example_ids(frame, mask, k=5):
    """Trả về tối đa k row_id (ngăn bằng '; ') của các dòng thỏa mask — dùng cho cột example_row_ids."""
    return '; '.join(frame.loc[mask, 'row_id'].astype(str).head(k))

def log_step(step, field, issue, detection_rule, action, affected_rows, changed_cells,
             removed_rows, example_row_ids, reason, check_after):
    """Thêm 1 dòng vào nhật ký làm sạch. Gọi lại với cùng step+field+issue sẽ GHI ĐÈ dòng cũ (an toàn khi chạy lại ô)."""
    row = dict(step=step, field=field, issue=issue, detection_rule=detection_rule, action=action,
               affected_rows=int(affected_rows), changed_cells=int(changed_cells),
               removed_rows=int(removed_rows), example_row_ids=example_row_ids,
               reason=reason, check_after=check_after)
    cleaning_log_rows[:] = [r for r in cleaning_log_rows
                            if not (r['step'] == step and r['field'] == field and r['issue'] == issue)]
    cleaning_log_rows.append(row)
    print(f'📝 Đã ghi nhật ký: [{step}] {field} — {issue} ({int(affected_rows)} dòng)')

def check(name, condition, hint=''):
    """In ✅ hoặc ❌. condition là hàm không tham số (lambda) để không báo lỗi khi biến chưa sẵn sàng."""
    try:
        ok = bool(condition())
        err = ''
    except Exception as e:
        ok, err = False, f' (chưa sẵn sàng: {type(e).__name__})'
    print(('✅ ' if ok else '❌ ') + name + ('' if ok else err + (f'\n   ↳ Gợi ý: {hint}' if hint else '')))
    return ok

def save_table(frame, name):
    """Lưu bảng nguồn vào outputs/tables/<name>.csv. Dùng đúng tên trong đề (Mục 12)."""
    if frame is None:
        print(f'⏳ Chưa có bảng {name} — bỏ qua.'); return
    if name + '.csv' not in TABLE_FILES:
        print(f'⚠️  Tên {name}.csv không có trong danh sách chuẩn: kiểm tra lại chính tả.')
    frame.to_csv(OUTPUT_DIR / 'tables' / f'{name}.csv', index=False, encoding='utf-8-sig')
    print(f'💾 outputs/tables/{name}.csv  ({len(frame)} dòng)')

def save_fig(fig, name):
    """Lưu hình PNG 150 dpi vào outputs/figures/<name>.png."""
    if name + '.png' not in FIGURE_FILES:
        print(f'⚠️  Tên {name}.png không có trong danh sách chuẩn: kiểm tra lại chính tả.')
    fig.savefig(OUTPUT_DIR / 'figures' / f'{name}.png', dpi=150, bbox_inches='tight')
    print(f'🖼️  outputs/figures/{name}.png')

def finish_chart(ax, title, xlabel, ylabel, note=None):
    """Định dạng chuẩn: tiêu đề, nhãn trục, ghi chú mẫu số/nguồn dưới hình, bỏ khung thừa."""
    ax.set_title(title, fontsize=13, fontweight='bold', loc='left')
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.spines[['top', 'right']].set_visible(False)
    if note:
        ax.figure.text(0.01, -0.04, note, fontsize=9, color='#555555', ha='left', va='top', wrap=True)

def show_progress():
    """Bảng tiến độ: các tệp bắt buộc đã có trong outputs/ hay chưa."""
    items = [('Dữ liệu', f, OUTPUT_DIR / f) for f in CORE_FILES] + \
            [('Bảng', f, OUTPUT_DIR / 'tables' / f) for f in TABLE_FILES] + \
            [('Hình', f, OUTPUT_DIR / 'figures' / f) for f in FIGURE_FILES]
    t = pd.DataFrame([(g, f, '✅' if p.exists() else '⬜') for g, f, p in items], columns=['Nhóm', 'Tệp', 'Có'])
    done = (t['Có'] == '✅').sum()
    print(f'Tiến độ tệp đầu ra: {done}/{len(t)} ({done / len(t):.0%})')
    return t

print('Đã nạp công cụ hỗ trợ. pandas', pd.__version__)

Đã nạp công cụ hỗ trợ. pandas 2.2.3


In [ ]:
# 0.2 ĐỌC DỮ LIỆU — đọc mọi cột ở dạng chuỗi để thấy giá trị gốc
if not DATA_PATH.exists():
    raise FileNotFoundError('Không thấy hotel_bookings_dirty.csv. Colab: tải tệp lên bảng Files. '
                            'Máy cá nhân: đặt tệp vào thư mục data/ hoặc sửa DATA_PATH.')
raw_df = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
print('Đường dẫn:', DATA_PATH, '| Kích thước:', raw_df.shape)
check('Đầu vào đúng 10.325 dòng × 24 cột', lambda: raw_df.shape == (10325, 24),
      'Bạn đang dùng sai tệp hoặc tệp đã bị sửa. Tải lại gói bài tập.')
raw_df.head()

Đường dẫn: hotel_bookings_dirty.csv | Kích thước: (10325, 24)
✅ Đầu vào đúng 10.325 dòng × 24 cột


,row_id,booking_id,hotel,is_canceled,lead_time,arrival_date,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,reserved_room_type,assigned_room_type,deposit_type,customer_type,adr,total_of_special_requests,reservation_status,reservation_status_date
0,R00001,BK046154,City Hotel,0,9,2015-12-18,0,1,2,0,0,SC,PRT,Online TA,TA/TO,0,A,A,No Deposit,Transient,85,1,Check-Out,2015-12-19
1,R00002,BK054785,City Hotel,1,177,2016-07-26,0,4,2,0,0,SC,ROU,Online TA,TA/TO,0,A,A,No Deposit,Transient,80.75,1,Canceled,2016-03-05
2,R00003,BK093598,City Hotel,0,174,2016-07-21,0,3,2,0,0,BB,GBR,Online TA,TA/TO,0,D,D,no deposit,Transient,115.8,2,Check-Out,2016-07-24
3,R00004,BK071930,City Hotel,1,163,2017-07-15,2,2,3,0,0,BB,DEU,Online TA,TA/TO,0,D,D,No Deposit,Transient,152.1,0,Canceled,2017-05-18
4,R00005,BK100632,City Hotel,0,1,"October 29, 2016",0,1,3,0,0,HB,BRA,Online TA,TA/TO,0,D,D,No Deposit,Transient,203.81,1,Check-Out,2016-10-30


---
# GIAI ĐOẠN A · KHẢO SÁT TRƯỚC KHI SỬA
*Mục tiêu của cả giai đoạn: hiểu dữ liệu đang sai ở đâu, sai bao nhiêu — **chưa sửa gì cả**.*

## A · Khảo sát dữ liệu

| | |
|---|---|
| **Mục tiêu** | Lập bảng thiếu theo cột và danh mục **ít nhất 10 nhóm vấn đề**, mỗi nhóm có số dòng và tối đa 5 row_id ví dụ. |
| **Đầu vào → Đầu ra** | `raw_df` → `missing_survey`, `issue_catalog` |
| **Thời gian gợi ý** | ≈ 2 giờ |
| **Đọc kèm** | Đề bài Mục 6 |

**Các bước cần làm**

1. **A1** In `shape`, tên cột, `dtypes`, 5 dòng mẫu.
2. **A2** Với từng cột: đếm ô rỗng (sau `strip`) và ô là ký hiệu thiếu (`MISSING_TOKENS`, không phân biệt hoa thường). Lưu thành `missing_survey` với các cột: `column, n_rows, empty_count, token_count, missing_total, missing_pct`.
3. **A3** `value_counts()` cho các cột phân loại: `hotel, is_canceled, meal, market_segment, distribution_channel, deposit_type, customer_type, reservation_status, reserved_room_type, assigned_room_type`.
4. **A4** Mẫu hóa định dạng của `arrival_date, reservation_status_date, lead_time, stays_in_*, adults, children, adr`; xem các cách ghi `country`.
5. **A5** Đếm dòng trùng trên **23 cột nghiệp vụ** (bỏ `row_id`); đếm `booking_id` lặp trước và sau khi `strip().upper()`.
6. **A6** Gom phát hiện vào `issue_catalog` (cột: `issue_id, field, issue, detection_rule, affected_rows, example_row_ids`).

**Gợi ý hàm:** `value_counts(dropna=False)`, `str.strip`, `str.lower`, `isin`, `duplicated(subset=...)`, `str.match`, `str.contains`

**⚠️ Bẫy thường gặp**

- Gọi `isna()` ngay sau `keep_default_na=False` → luôn ra 0 vì ô trống đang là chuỗi `''`.
- Gọi `duplicated()` trên cả 24 cột → ra 0 vì `row_id` luôn khác nhau.
- Coi hai đặt phòng **giống hệt mọi cột trừ `booking_id`** là trùng — một đoàn khách có thể đặt nhiều phòng y hệt nhau.

💡 **Mẹo nhìn định dạng nhanh:** thay mọi chữ số bằng `9` rồi đếm mẫu, ví dụ `raw_df['adr'].str.replace(r'\d', '9', regex=True).value_counts()` — các cách viết lạ (ký hiệu tiền, dấu phẩy thập phân, số âm…) sẽ lộ ra ngay.

In [ ]:
# A1 · Tổng quan
# TODO: in raw_df.shape, list(raw_df.columns), raw_df.dtypes và raw_df.head()
print(raw_df.shape)
print(list(raw_df.columns))
print(raw_df.dtypes)

(10325, 24)
['row_id', 'booking_id', 'hotel', 'is_canceled', 'lead_time', 'arrival_date', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'adr', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']
row_id                       object
booking_id                   object
hotel                        object
is_canceled                  object
lead_time                    object
arrival_date                 object
stays_in_weekend_nights      object
stays_in_week_nights         object
adults                       object
children                     object
babies                       object
meal                         object
country                      object
market_segment               object
distribution_channel         object
is_repeated_guest            object
reserve

In [ ]:
# A2 · Bảng thiếu theo cột
# TODO: DataFrame 24 dòng, cột: column, n_rows, empty_count, token_count, missing_total, missing_pct

tokens_lower = [t.lower() for t in MISSING_TOKENS]
n_total = len(raw_df)
rows = []

for col in raw_df.columns:
    s_raw = raw_df[col]
    s_str = s_raw.astype(str).str.strip()

    # 1. Ô rỗng, NaN hoặc chỉ có khoảng trắng
    is_empty = _blank(s_raw)
    # 2. Ô chứa các ký hiệu thiếu (không phân biệt hoa thường, ngoại trừ các ô đã trống ở trên)
    # ~: đảo kết quả => ~is_empty trả về những ô ko rỗng (xét các dữ liệu nan, null, ...)
    is_token = ~is_empty & s_str.str.lower().isin(tokens_lower)

    empty_cnt = int(is_empty.sum())
    token_cnt = int(is_token.sum())
    missing_tot = empty_cnt + token_cnt
    missing_pct = round((missing_tot / n_total) * 100, 2)

    rows.append({
        'column': col,
        'n_rows': n_total,
        'empty_count': empty_cnt,
        'token_count': token_cnt,
        'missing_total': missing_tot,
        'missing_pct': missing_pct
    })

missing_survey = pd.DataFrame(rows)
missing_survey

,column,n_rows,empty_count,token_count,missing_total,missing_pct
0,row_id,10325,0,0,0,0.00
1,booking_id,10325,0,0,0,0.00
2,hotel,10325,0,0,0,0.00
3,is_canceled,10325,0,0,0,0.00
4,lead_time,10325,0,10,10,0.10
5,arrival_date,10325,25,0,25,0.24
6,stays_in_weekend_nights,10325,11,0,11,0.11
7,stays_in_week_nights,10325,10,0,10,0.10
8,adults,10325,12,0,12,0.12
9,children,10325,26,15,41,0.40


In [ ]:
# A3 · Nhãn của các cột phân loại
# TODO: value_counts cho từng cột phân loại
CAT_COLS = [
    'hotel', 'is_canceled', 'meal', 'market_segment', 'distribution_channel',
    'deposit_type', 'customer_type', 'reservation_status',
    'reserved_room_type', 'assigned_room_type']

for col in CAT_COLS:
 print(raw_df[col].value_counts(dropna=False))

hotel
City Hotel      6763
Resort Hotel    3341
city hotel        64
resort hotel      34
CITY HOTEL        31
City              30
 City Hotel       26
Resort            25
RESORT HOTEL      11
Name: count, dtype: int64
is_canceled
0        6351
1        3769
yes        51
no         50
True       33
False      30
1.0        21
0.0        20
Name: count, dtype: int64
meal
BB            7830
HB            1250
SC             922
Undefined      106
FB              59
bb              56
B&B             30
Half Board      26
                24
sc              13
Full Board       8
hb               1
Name: count, dtype: int64
market_segment
Online TA        4808
Offline TA/TO    2052
Groups           1732
Direct           1052
Corporate         454
Complementary      63
online ta          51
Online_TA          32
Aviation           23
offline ta/to      20
GROUPS             15
direct             10
OnlineTA           10
Undefined           3
Name: count, dtype: int64
distribution_channel


In [ ]:
# A4 · Định dạng ngày, số, tiền và mã quốc gia
# TODO: mẫu hóa định dạng (thay chữ số bằng 9) và value_counts

In [ ]:
# A5 · Bản ghi trùng và khóa
# TODO: đếm trùng trên 23 cột nghiệp vụ; đếm booking_id lặp trước/sau chuẩn hóa; lưu vài ví dụ row_id

In [ ]:
# A6 · Danh mục vấn đề (≥ 10 nhóm)
issue_catalog = None    # TODO: DataFrame cột: issue_id, field, issue, detection_rule, affected_rows, example_row_ids

In [ ]:
# ✅ Tự kiểm tra giai đoạn A
check('A2 missing_survey có 24 dòng (1 dòng/cột)', lambda: len(missing_survey) == 24)
check('A2 missing_survey đủ cột', lambda: {'column','n_rows','empty_count','token_count','missing_total','missing_pct'} <= set(missing_survey.columns))
check('A6 issue_catalog có ≥ 10 nhóm vấn đề', lambda: len(issue_catalog) >= 10, 'Xem lại gợi ý trong thẻ nhiệm vụ A.')
check('A6 mỗi nhóm có ≤ 5 row_id ví dụ', lambda: issue_catalog['example_row_ids'].str.split('; ').str.len().le(5).all())
check('raw_df vẫn nguyên vẹn', lambda: raw_df.shape == (10325, 24))
save_table(missing_survey, 'a_missing_survey')
save_table(issue_catalog, 'a_issue_catalog')

> **✍️ Phát hiện chính và quy tắc dự kiến** *(bắt buộc — 3 đến 6 câu)*
>
> - Quy tắc nhóm đã áp dụng: …
> - Vì sao chọn cách này thay vì cách khác: …
> - Ảnh hưởng đến kết quả phân tích phía sau: …

---
# GIAI ĐOẠN B · LÀM SẠCH (B1 → B8)

**Nguyên tắc chung cho mọi bước**

1. Làm trên `df` (bản sao); **không bao giờ sửa `raw_df`**.
2. Mỗi quy tắc → **gọi `log_step(...)` một lần** với số liệu đếm được bằng mã, không gõ tay.
3. Giá trị không sửa được bằng bằng chứng có sẵn → **để thiếu (`pd.NA`)**, không đoán, không điền giá trị phổ biến nhất hay trung bình.
4. Đếm **trước và sau** mỗi bước để điền `check_after`.

| Cột nhật ký | Ý nghĩa ngắn |
|---|---|
| `affected_rows` | Số dòng **thỏa điều kiện phát hiện** |
| `changed_cells` | Số **ô thực sự đổi giá trị** |
| `removed_rows` | Số dòng **bị loại khỏi bảng** (thường chỉ khác 0 ở bước trùng/gộp) |

In [ ]:
# Bản làm việc — chạy lại ô này nếu muốn làm lại giai đoạn B từ đầu
df = raw_df.copy()
cleaning_log_rows.clear()
print('df:', df.shape)

## B1 · Khóa và bản ghi trùng tuyệt đối

| | |
|---|---|
| **Mục tiêu** | Phân biệt `row_id` (dòng nhập) với `booking_id` (đặt phòng); xử lý trùng tuyệt đối 23 cột. |
| **Đầu vào → Đầu ra** | `df` → `df` có `booking_id` đã chuẩn hóa |
| **Thời gian gợi ý** | ≈ 30 phút |
| **Đọc kèm** | Đề bài Mục 7 · B1 |

**Các bước cần làm**

1. Chuẩn hóa `booking_id`: bỏ khoảng trắng đầu/cuối, chuyển **chữ hoa** (dạng chuẩn `BK` + 6 chữ số).
2. Đếm dòng trùng trên 23 cột nghiệp vụ; lưu ví dụ **trước** khi xử lý.
3. Quyết định: loại ngay các dòng trùng tuyệt đối, hoặc để B8 gộp. Ghi rõ trong nhật ký. Nếu loại ngay, giữ danh sách row_id bị loại để lập `row_lineage` ở B8.

**Gợi ý hàm:** `str.strip().str.upper()`, `duplicated(subset=..., keep='first')`, `str.fullmatch(r'BK\d{6}')`

**⚠️ Bẫy thường gặp**

- Gộp hai đặt phòng khác `booking_id` chỉ vì các cột còn lại giống nhau.
- Xóa dòng trùng mà không lưu row_id → mất truy vết.

In [ ]:
# TODO B1

In [ ]:
check('B1 booking_id đúng dạng BK + 6 chữ số', lambda: df['booking_id'].str.fullmatch(r'BK\d{6}').all())

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - Quy tắc nhóm đã áp dụng: …
> - Vì sao chọn cách này thay vì cách khác: …
> - Ảnh hưởng đến kết quả phân tích phía sau: …

## B2 · Giá trị nằm nhầm cột

| | |
|---|---|
| **Mục tiêu** | Chuyển ngày nằm nhầm ở `reservation_status` sang `reservation_status_date` **khi đủ bằng chứng**. |
| **Đầu vào → Đầu ra** | `df` → `df` |
| **Thời gian gợi ý** | ≈ 20 phút |
| **Đọc kèm** | Đề bài Mục 7 · B2 |

**Các bước cần làm**

1. Tìm dòng thỏa **cả 2 điều kiện**: `reservation_status` (sau strip) có dạng ngày `YYYY-MM-DD` **và** `reservation_status_date` đang rỗng.
2. Chỉ với các dòng đó: chuyển giá trị sang `reservation_status_date`, để `reservation_status` rỗng, ghi nhật ký.
3. Làm bước này **trước** khi chuẩn hóa nhãn (B4): nếu ánh xạ nhãn trước, các ngày nằm nhầm sẽ bị biến thành thiếu và mất bằng chứng.

**Gợi ý hàm:** `str.strip()`, `str.fullmatch(r'\d{4}-\d{2}-\d{2}')`, toán tử `&`

**⚠️ Bẫy thường gặp**

- Ghi đè `reservation_status_date` đang có dữ liệu.
- Đoán trạng thái cho các dòng vừa chuyển — trạng thái để thiếu.

In [ ]:
# TODO B2

In [ ]:
check('B2 reservation_status không còn chứa ngày', lambda: not df['reservation_status'].astype(str).str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}').any())

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - Quy tắc nhóm đã áp dụng: …
> - Vì sao chọn cách này thay vì cách khác: …
> - Ảnh hưởng đến kết quả phân tích phía sau: …

## B3 · Giá trị thiếu theo từng cột

| | |
|---|---|
| **Mục tiêu** | Đưa các ký hiệu thiếu về `pd.NA`; giữ thiếu trong dữ liệu. |
| **Đầu vào → Đầu ra** | `df` → `df` |
| **Thời gian gợi ý** | ≈ 30 phút |
| **Đọc kèm** | Đề bài Mục 7 · B3 |

**Các bước cần làm**

1. Ô rỗng `''` và `MISSING_TOKENS` (`NULL`, `N/A`, `NA`, `?`, `unknown`, `-`) → `pd.NA` ở mọi cột trừ `row_id`, `booking_id`.
2. `Undefined` ở `market_segment`, `distribution_channel` → thiếu (nguồn dùng nhãn này cho *không xác định*).
3. Ghi một dòng nhật ký cho mỗi cột có thay đổi (hoặc một dòng chung nhưng nêu đủ cột).

**Gợi ý hàm:** `str.lower().isin([t.lower() for t in MISSING_TOKENS])`, `mask`, `replace`

**⚠️ Bẫy thường gặp**

- Điền `children`, `country`, `adr` bằng giá trị phổ biến/trung bình (bị cấm).
- Xóa mọi dòng có ô thiếu (bị cấm) — một đặt phòng thiếu quốc gia vẫn dùng được khi tính tỷ lệ hủy theo khách sạn.

In [ ]:
# TODO B3

In [ ]:
check('B3 không còn ký hiệu thiếu dạng chữ',
      lambda: not any(df[c].dropna().astype(str).str.lower().isin([t.lower() for t in MISSING_TOKENS]).any()
                      for c in df.columns if c not in ('row_id', 'booking_id')))
check('B3 không còn chuỗi rỗng', lambda: not (df.drop(columns='row_id').astype('string') == '').any().any())

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - Quy tắc nhóm đã áp dụng: …
> - Vì sao chọn cách này thay vì cách khác: …
> - Ảnh hưởng đến kết quả phân tích phía sau: …

## B4 · Chuẩn hóa nhãn phân loại và nhất quán trạng thái

| | |
|---|---|
| **Mục tiêu** | Đưa mọi cột phân loại về đúng tập nhãn chuẩn (Mục 5.2) và làm `is_canceled` nhất quán với `reservation_status` (Mục 5.3). |
| **Đầu vào → Đầu ra** | `df` → `df` |
| **Thời gian gợi ý** | ≈ 45 phút |
| **Đọc kèm** | Đề bài Mục 5.2, 5.3 và Mục 7 · B4 |

**Các bước cần làm**

1. Mọi cột (trừ `row_id`): `strip` + gom nhiều khoảng trắng thành 1.
2. Lập từ điển ánh xạ cho `hotel, meal, market_segment, distribution_channel, deposit_type, customer_type, reservation_status`. In `value_counts` **trước và sau**.
3. `is_canceled`, `is_repeated_guest`: đưa `yes/no`, `True/False`, `1.0/0.0` về số 0/1 (kiểu `Int64`).
4. `reserved_room_type`, `assigned_room_type`: chữ hoa; mã ngoài tập hợp lệ → thiếu.
5. Đếm số dòng `is_canceled` **mâu thuẫn** với `reservation_status` (Check-Out ↔ 0; Canceled, No-Show ↔ 1). Khi `reservation_status` hợp lệ, **tính lại** `is_canceled` từ nó; khi trạng thái thiếu, giữ `is_canceled` đã chuẩn hóa. Ghi số dòng bị sửa.

**Gợi ý hàm:** `str.lower().str.replace(r'[^a-z]', '', regex=True)` để tạo khóa so khớp, `map(dict)`, `isin`

**⚠️ Bẫy thường gặp**

- `map` với từ điển thiếu khóa sẽ biến giá trị đó thành NaN — luôn kiểm tra số NaN mới phát sinh sau khi `map`.
- Đổi nhãn `Undefined` của `meal` thành thiếu — theo tài liệu nguồn, `Undefined` và `SC` cùng nghĩa *không kèm bữa ăn* (xem Mục 5.2).

In [ ]:
HOTEL_MAP = {
    # TODO: 'cityhotel': 'City Hotel', ...   (khóa: chữ thường, chỉ giữ chữ cái)
}
MEAL_MAP = {}       # TODO
SEGMENT_MAP = {}    # TODO
CHANNEL_MAP = {}    # TODO
DEPOSIT_MAP = {}    # TODO
CUSTOMER_MAP = {}   # TODO
STATUS_MAP = {}     # TODO
# TODO B4

In [ ]:
check('B4 hotel chỉ còn City Hotel / Resort Hotel và không thiếu', lambda: set(df['hotel']) <= set(HOTELS) and df['hotel'].notna().all())
check('B4 meal hợp lệ hoặc thiếu', lambda: df['meal'].dropna().isin(MEALS).all())
check('B4 market_segment hợp lệ hoặc thiếu', lambda: df['market_segment'].dropna().isin(SEGMENTS).all())
check('B4 distribution_channel hợp lệ hoặc thiếu', lambda: df['distribution_channel'].dropna().isin(CHANNELS).all())
check('B4 deposit_type, customer_type hợp lệ hoặc thiếu',
      lambda: df['deposit_type'].dropna().isin(DEPOSITS).all() and df['customer_type'].dropna().isin(CUSTOMERS).all())
check('B4 room type hợp lệ hoặc thiếu',
      lambda: df['reserved_room_type'].dropna().isin(ROOM_TYPES).all() and df['assigned_room_type'].dropna().isin(ROOM_TYPES).all())
check('B4 is_canceled, is_repeated_guest kiểu Int64 (0/1)',
      lambda: str(df['is_canceled'].dtype) == 'Int64' and str(df['is_repeated_guest'].dtype) == 'Int64')
check('B4 reservation_status hợp lệ hoặc thiếu', lambda: df['reservation_status'].dropna().isin(STATUSES).all())
check('B4 is_canceled nhất quán với reservation_status',
      lambda: (df['reservation_status'].map({'Check-Out': 0, 'Canceled': 1, 'No-Show': 1}).dropna()
               == df.loc[df['reservation_status'].notna(), 'is_canceled']).all())

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - Quy tắc nhóm đã áp dụng: …
> - Vì sao chọn cách này thay vì cách khác: …
> - Ảnh hưởng đến kết quả phân tích phía sau: …

## B5 · Ngày đến và ngày cập nhật trạng thái

| | |
|---|---|
| **Mục tiêu** | Chuyển `arrival_date` (4 định dạng, dạng `/` luôn đọc ngày/tháng/năm) và `reservation_status_date` sang kiểu ngày, đúng miền. |
| **Đầu vào → Đầu ra** | `df` → `df` với 2 cột kiểu datetime |
| **Thời gian gợi ý** | ≈ 45 phút |
| **Đọc kèm** | Đề bài Mục 5.1 và Mục 7 · B5 |

**Các bước cần làm**

1. Đọc **từng định dạng** của `arrival_date` bằng `pd.to_datetime(..., format=..., errors='coerce')` rồi ghép kết quả.
2. `arrival_date` ngoài 01/07/2015 – 31/08/2017, ngày không tồn tại, hoặc không đọc được → `NaT`; ghi lý do.
3. `reservation_status_date` (dạng ISO): ngoài 01/10/2014 – 14/09/2017 hoặc không tồn tại → `NaT`.
4. Không tự đổi ngày sai thành ngày gần nhất hợp lệ.

**Gợi ý hàm:** `pd.to_datetime(format='%Y-%m-%d')`, `'%d/%m/%Y'`, `'%B %d, %Y'`, `'%d-%b-%Y'`, `fillna`, `between`

**⚠️ Bẫy thường gặp**

- Để pandas tự đoán định dạng → `05/03/2016` có thể bị đọc thành 3 tháng 5.
- Xử lý `reservation_status_date` trước khi chuyển các ngày nằm nhầm ở `reservation_status` (B5) — nên làm B5 phần chuyển ô trước, hoặc chạy lại B4 sau B5.

In [ ]:
DATE_FORMATS = [
    # TODO: liệt kê 4 chuỗi format của arrival_date
]
# TODO B5

In [ ]:
check('B5 arrival_date là kiểu datetime, trong miền', lambda: pd.api.types.is_datetime64_any_dtype(df['arrival_date'])
      and df['arrival_date'].dropna().between(ARRIVAL_MIN, ARRIVAL_MAX).all())
check('B5 reservation_status_date là kiểu datetime, trong miền', lambda: pd.api.types.is_datetime64_any_dtype(df['reservation_status_date'])
      and df['reservation_status_date'].dropna().between(STATUS_MIN, STATUS_MAX).all())

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - Quy tắc nhóm đã áp dụng: …
> - Vì sao chọn cách này thay vì cách khác: …
> - Ảnh hưởng đến kết quả phân tích phía sau: …

## B6 · Số đo: lead_time, số đêm, số khách, yêu cầu đặc biệt và giá phòng (adr)

| | |
|---|---|
| **Mục tiêu** | Chuyển các cột số sang kiểu đúng (`Int64` cho số nguyên, `float` cho `adr`), đúng miền. |
| **Đầu vào → Đầu ra** | `df` → `df` |
| **Thời gian gợi ý** | ≈ 45 phút |
| **Đọc kèm** | Đề bài Mục 5.1 và Mục 7 · B6 |

**Các bước cần làm**

1. Số nguyên: `lead_time` (0–800), `stays_in_weekend_nights` (0–20), `stays_in_week_nights` (0–50), `adults`, `children`, `babies` (0–10), `total_of_special_requests` (0–5).
2. Chấp nhận `2.0` → 2 và hậu tố đơn vị **đúng nghĩa cột** (`3 nights` ở cột số đêm → 3). Giá trị âm, không phải số, ngoài miền → thiếu + lý do.
3. `adr` (EUR/đêm): bỏ `€`, `EUR`; đổi dấu phẩy thập phân `95,50` → `95.50`; miền 0–1000. **`adr = 0` hợp lệ** (phòng tặng, gói đặc biệt) — giữ nguyên.
4. Giá trị hiếm nhưng trong miền (ví dụ ở 1 đêm, đặt trước 600 ngày, 0 khách) → **giữ**, chỉ báo cáo số lượng.

**Gợi ý hàm:** `str.replace(r'\s*nights?$', '', regex=True)`, `pd.to_numeric(errors='coerce')`, `where`, `between`, `.astype('Int64')`

**⚠️ Bẫy thường gặp**

- Dùng `int` thường → lỗi khi có thiếu; phải dùng `Int64`.
- Xóa `adr = 0` hoặc giá phòng cao (trong miền) chỉ vì khác số đông.
- Đổi `95,50` thành 9550 do xóa dấu phẩy thay vì đổi thành dấu chấm.

In [ ]:
INT_RULES = {
    # TODO: 'lead_time': (0, 800), ...
}
# TODO B6

In [ ]:
check('B6 các cột số nguyên có kiểu Int64 và đúng miền',
      lambda: all(str(df[c].dtype) == 'Int64' and df[c].dropna().between(lo, hi).all() for c, lo, hi in [
          ('lead_time', LEAD_MIN, LEAD_MAX), ('stays_in_weekend_nights', 0, WEEKEND_MAX), ('stays_in_week_nights', 0, WEEK_MAX),
          ('adults', 0, GUEST_MAX), ('children', 0, GUEST_MAX), ('babies', 0, GUEST_MAX), ('total_of_special_requests', 0, REQ_MAX)]))
check('B6 adr là số thực trong 0–1000', lambda: pd.api.types.is_float_dtype(df['adr']) and df['adr'].dropna().between(ADR_MIN, ADR_MAX).all())

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - Quy tắc nhóm đã áp dụng: …
> - Vì sao chọn cách này thay vì cách khác: …
> - Ảnh hưởng đến kết quả phân tích phía sau: …

## B7 · Mã quốc gia

| | |
|---|---|
| **Mục tiêu** | `country` là mã quốc gia viết hoa theo nguồn (ISO 3 chữ cái); tên đầy đủ được đổi về mã. |
| **Đầu vào → Đầu ra** | `df` → `df` |
| **Thời gian gợi ý** | ≈ 20 phút |
| **Đọc kèm** | Đề bài Mục 5.1 và Mục 7 · B7 |

**Các bước cần làm**

1. Viết hoa, bỏ khoảng trắng.
2. Đổi tên đầy đủ về mã theo bảng quy ước: `Portugal` → `PRT`; `United Kingdom`, `UK` → `GBR`; `Spain` → `ESP`; `France` → `FRA`; `Germany` → `DEU`.
3. Giá trị không phải 2–3 chữ cái in hoa → thiếu + ghi nhật ký.
4. Giữ nguyên mã `CN` của nguồn — **không** tự gộp với `CHN` khi chưa có bằng chứng (nêu trong giới hạn).

**Gợi ý hàm:** `str.upper()`, `replace(dict)`, `str.fullmatch(r'[A-Z]{2,3}')`, `where`

**⚠️ Bẫy thường gặp**

- Ánh xạ chuỗi con thay vì cả giá trị.
- Đoán quốc gia từ khách sạn hay kênh bán.

In [ ]:
COUNTRY_MAP = {
    # TODO: 'PORTUGAL': 'PRT', ...
}
# TODO B7

In [ ]:
check('B7 country là mã 2–3 chữ in hoa hoặc thiếu', lambda: df['country'].dropna().str.fullmatch(r'[A-Z]{2,3}').all())

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - Quy tắc nhóm đã áp dụng: …
> - Vì sao chọn cách này thay vì cách khác: …
> - Ảnh hưởng đến kết quả phân tích phía sau: …

## B8 · Hợp nhất các dòng cùng booking_id và lập truy vết

| | |
|---|---|
| **Mục tiêu** | Mỗi `booking_id` còn **đúng 1 dòng**; mọi `row_id` gốc truy được về `booking_id` cuối cùng. |
| **Đầu vào → Đầu ra** | `df` → `merged_df`, `row_lineage` |
| **Thời gian gợi ý** | ≈ 45 phút |
| **Đọc kèm** | Đề bài Mục 7 · B8 |

**Các bước cần làm**

1. Với các dòng cùng `booking_id`, so sánh **giá trị không thiếu** theo từng cột.
2. Giá trị đồng thuận → giữ. Một bên thiếu, một bên có → lấy giá trị có. **Mâu thuẫn** → áp dụng quy tắc nhóm tự chọn (ví dụ: lấy từ dòng có `row_id` nhỏ nhất) và lưu danh sách xung đột.
3. Tạo `row_lineage` với cột `row_id, booking_id, status` (`kept`, `merged`, `dropped`).
4. Ghi nhật ký: số dòng trước, số dòng sau, số dòng loại, số booking_id có xung đột.

**Gợi ý hàm:** `groupby('booking_id')`, `nunique(dropna=True)`, `first()` (bỏ qua NaN), `duplicated`

**⚠️ Bẫy thường gặp**

- Gộp trước khi chuẩn hóa xong → bỏ sót trùng hoặc tạo xung đột giả (ví dụ `95,50` và `95.5`).
- Quên các dòng đã loại ở B1 trong `row_lineage`.

In [ ]:
merged_df = None     # TODO: 1 dòng / booking_id
row_lineage = None   # TODO: cột row_id, booking_id, status

In [ ]:
check('B8 merged_df: booking_id duy nhất', lambda: merged_df['booking_id'].is_unique)
check('B8 row_lineage phủ đủ 10.325 row_id', lambda: set(row_lineage['row_id']) == set(raw_df['row_id']))
check('B8 số dòng kept = số đặt phòng', lambda: (row_lineage['status'] == 'kept').sum() == len(merged_df))

> **✍️ Quyết định và lý do của nhóm** *(bắt buộc — 3 đến 6 câu)*
>
> - Quy tắc nhóm đã áp dụng: …
> - Vì sao chọn cách này thay vì cách khác: …
> - Ảnh hưởng đến kết quả phân tích phía sau: …

---
# GIAI ĐOẠN C · BẢNG SẠCH, BẢNG THÁNG VÀ KIỂM TRA

## C1 · Cột dẫn xuất

| | |
|---|---|
| **Mục tiêu** | Hoàn thiện `clean_df` (1 dòng/booking_id) với 7 cột dẫn xuất. |
| **Đầu vào → Đầu ra** | `merged_df` → `clean_df` |
| **Thời gian gợi ý** | ≈ 45 phút |
| **Đọc kèm** | Đề bài Mục 8.1 |

**Các bước cần làm**

1. `arrival_year`, `arrival_month` từ `arrival_date` hợp lệ.
2. `total_nights` = weekend + week nights; `total_guests` = adults + children + babies (thiếu một thành phần → kết quả thiếu).
3. `lead_time_group`: `0-7`, `8-30`, `31-90`, `91-180`, `181+` (ngày).
4. `room_changed_flag` = True khi cả hai loại phòng có dữ liệu và khác nhau; còn lại False.
5. `revenue_est` = `adr × total_nights` **chỉ khi** `is_canceled == 0` và hai giá trị hợp lệ; còn lại thiếu. Làm tròn 2 chữ số.
6. Xuất hai cột ngày dạng chuỗi `YYYY-MM-DD`.

**Gợi ý hàm:** `dt.year`, `dt.month`, `dt.strftime('%Y-%m-%d')`, `pd.cut(bins=[-0.5, 7.5, 30.5, 90.5, 180.5, float('inf')])`, `where`, `round`

**⚠️ Bẫy thường gặp**

- Coi `revenue_est` là doanh thu thực — đây chỉ là **ước tính tiền phòng** từ giá bình quân/đêm.
- Điền 0 cho `total_guests` khi thiếu `children`.

In [ ]:
clean_df = None      # TODO: đủ các cột trong CLEAN_REQUIRED (có thể thêm quality_flags)

## C2 · Bảng tổng hợp theo tháng

| | |
|---|---|
| **Mục tiêu** | Tạo `monthly_summary`: mỗi dòng là một cặp (khách sạn, năm, tháng đến). |
| **Đầu vào → Đầu ra** | `clean_df` → `monthly_summary` |
| **Thời gian gợi ý** | ≈ 30 phút |
| **Đọc kèm** | Đề bài Mục 8.2 |

**Các bước cần làm**

1. Chỉ dùng đặt phòng có `arrival_date` hợp lệ.
2. `n_bookings` = số đặt phòng; `n_canceled` = số có `is_canceled == 1`.
3. `cancel_rate_pct` = `n_canceled` / số đặt phòng có `is_canceled` không thiếu × 100, làm tròn 2 chữ số.
4. `room_nights` = tổng `total_nights` của đặt phòng **không hủy**.
5. Đúng 7 cột theo thứ tự: `hotel, arrival_year, arrival_month, n_bookings, n_canceled, cancel_rate_pct, room_nights`.

**Gợi ý hàm:** `groupby([...]).agg(...)`, `reset_index`, `round(2)`

**⚠️ Bẫy thường gặp**

- Lấy số dòng của `monthly_summary` làm số đặt phòng.
- Chia cho `n_bookings` khi có đặt phòng thiếu `is_canceled`.

In [ ]:
monthly_summary = None   # TODO

## C3 · Nhật ký và bảng chất lượng trước/sau

| | |
|---|---|
| **Mục tiêu** | Hoàn tất `cleaning_log` từ các lần gọi `log_step` và lập `quality_report`. |
| **Đầu vào → Đầu ra** | `cleaning_log_rows`, `raw_df`, `clean_df` → `cleaning_log`, `quality_report` |
| **Thời gian gợi ý** | ≈ 30 phút |
| **Đọc kèm** | Đề bài Mục 8.3 |

**Các bước cần làm**

1. `cleaning_log = pd.DataFrame(cleaning_log_rows, columns=LOG_COLUMNS)` — đã có sẵn bên dưới.
2. `quality_report` dạng dài: mỗi cột × `stage` (`before` / `after`), với `denominator_rows` (10.325 hoặc số đặt phòng), `missing_count`, `invalid_count`, `missing_pct`, có thể thêm `note`.
3. Giải thích được vì sao tỷ lệ thiếu của một số cột **tăng** sau làm sạch.

**Gợi ý hàm:** `pd.DataFrame(list_of_dicts)`, `pd.concat`

**⚠️ Bẫy thường gặp**

- Dùng chung một mẫu số cho trước và sau khi số dòng đã thay đổi.

In [ ]:
cleaning_log = pd.DataFrame(cleaning_log_rows, columns=LOG_COLUMNS) if cleaning_log_rows else None
quality_report = None   # TODO: cột column, stage, denominator_rows, missing_count, invalid_count, missing_pct

## C4 · Chạy bộ kiểm tra toàn diện

Ô dưới chạy **cùng bộ kiểm tra** với công cụ `kiem_tra_nop_bai.py`. Mục tiêu: **tất cả ✅** trước khi sang giai đoạn D.
Nếu một mục ❌, đọc phần chi tiết để biết số dòng sai, sửa ở bước B/C tương ứng rồi chạy lại.

In [ ]:
report = validate_outputs(clean_df, monthly_summary, row_lineage, cleaning_log, quality_report,
                          raw_row_ids=raw_df['row_id'])
all_ok = print_report(report)

---
# GIAI ĐOẠN D · TRẢ LỜI 10 CÂU HỎI

**Quy tắc chung cho mọi câu**

- Phân tích từ `clean_df` (đơn vị: **1 đặt phòng = 1 booking_id**).
- **Tỷ lệ hủy** = số đặt phòng `is_canceled == 1` / số đặt phòng có `is_canceled` không thiếu. Ghi rõ mẫu số.
- Mỗi câu: **bảng nguồn** (`save_table`) → **biểu đồ** (`save_fig`, nếu được yêu cầu) → **nhận xét ✍️**.
- Màu theo `HOTEL_COLORS` / `STATUS_COLORS`; trục số lượng và tỷ lệ của biểu đồ cột bắt đầu từ 0; ghi `n = …` trong ghi chú hình.
- Dùng `finish_chart(ax, title, xlabel, ylabel, note)` để định dạng thống nhất.

## Câu 1 · Chất lượng dữ liệu đã thay đổi thế nào

| | |
|---|---|
| **Cần tính** | So sánh trước/sau: số dòng, số booking_id, số dòng trùng, tỷ lệ thiếu từng cột. Nêu ≥ 3 quyết định làm sạch ảnh hưởng tới kết quả và **một chỉ tiêu bị sai nếu chưa chuẩn hóa** (ví dụ tỷ lệ hủy tính trên cột `is_canceled` thô, hoặc số đặt phòng của City Hotel khi chưa gộp biến thể tên). |
| **Mẫu số và trình bày** | Mẫu số: 10.325 dòng (trước) và số đặt phòng (sau). Không bắt buộc biểu đồ. |
| **Đầu ra** | `clean_df` → `q01_before_after` |

**Gợi ý:** Cột gợi ý: `metric, before, after, note`.

In [ ]:
q01 = None   # TODO: bảng q01_before_after

save_table(q01, 'q01_before_after')

In [ ]:
check('q01_before_after đã được tạo', lambda: q01 is not None and len(q01) > 0)

> **✍️ Nhận xét Câu 1** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt phòng (đã loại … đặt phòng vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…`*

## Câu 2 · Số đặt phòng và tỷ lệ hủy theo khách sạn

| | |
|---|---|
| **Cần tính** | Số đặt phòng, tỷ trọng mỗi khách sạn, số hủy và **tỷ lệ hủy** theo `hotel`; chênh lệch tỷ lệ hủy tính bằng **điểm phần trăm**. |
| **Mẫu số và trình bày** | Biểu đồ 1: cột (tỷ lệ hủy hoặc số đặt phòng tách Hủy/Không hủy), ghi số trên cột. |
| **Đầu ra** | `clean_df` → `q02_hotel_cancel` + `fig01_hotel_cancel.png` |

**Gợi ý:** Cột gợi ý: `hotel, n_bookings, share_pct, n_canceled, cancel_rate_pct`.

In [ ]:
q02 = None   # TODO: bảng q02_hotel_cancel

save_table(q02, 'q02_hotel_cancel')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt phòng; loại … vì …. Nguồn: outputs/tables/q02_hotel_cancel.csv')
# save_fig(fig, 'fig01_hotel_cancel')
# plt.show()

In [ ]:
check('q02_hotel_cancel đã được tạo', lambda: q02 is not None and len(q02) > 0)
check('q02 có đúng 2 khách sạn', lambda: len(q02) == 2)
check('Đã lưu fig01_hotel_cancel.png', lambda: (OUTPUT_DIR / 'figures' / 'fig01_hotel_cancel.png').exists())

> **✍️ Nhận xét Câu 2** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt phòng (đã loại … đặt phòng vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig01_hotel_cancel.png`*

## Câu 3 · Mùa vụ: số đặt phòng theo tháng đến năm 2016

| | |
|---|---|
| **Cần tính** | Chỉ năm **2016** (năm duy nhất đủ 12 tháng trong dữ liệu): số đặt phòng theo tháng × khách sạn; tỷ lệ hủy theo tháng. Nêu tháng cao nhất, thấp nhất của mỗi khách sạn. |
| **Mẫu số và trình bày** | Biểu đồ 2: hai đường theo tháng 1–12. Có thể dùng `monthly_summary`. Không so sánh trực tiếp với 2015, 2017 (thiếu tháng). |
| **Đầu ra** | `clean_df` → `q03_monthly_arrivals_2016` + `fig02_monthly_arrivals_2016.png` |

**Gợi ý:** Cột gợi ý: `arrival_month, City Hotel, Resort Hotel, total, cancel_rate_pct`.

In [ ]:
q03 = None   # TODO: bảng q03_monthly_arrivals_2016

save_table(q03, 'q03_monthly_arrivals_2016')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt phòng; loại … vì …. Nguồn: outputs/tables/q03_monthly_arrivals_2016.csv')
# save_fig(fig, 'fig02_monthly_arrivals_2016')
# plt.show()

In [ ]:
check('q03_monthly_arrivals_2016 đã được tạo', lambda: q03 is not None and len(q03) > 0)
check('q03 có đủ 12 tháng', lambda: len(q03) == 12)
check('Đã lưu fig02_monthly_arrivals_2016.png', lambda: (OUTPUT_DIR / 'figures' / 'fig02_monthly_arrivals_2016.png').exists())

> **✍️ Nhận xét Câu 3** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt phòng (đã loại … đặt phòng vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig02_monthly_arrivals_2016.png`*

## Câu 4 · Thời gian đặt trước (lead time) và hủy phòng

| | |
|---|---|
| **Cần tính** | Thống kê `lead_time` theo khách sạn (số mẫu, trung bình, trung vị, Q1, Q3, max) và **tỷ lệ hủy theo `lead_time_group`**. |
| **Mẫu số và trình bày** | Biểu đồ 3: cột tỷ lệ hủy theo 5 nhóm, **đúng thứ tự** `0-7 → 181+`. Đây là mối liên hệ quan sát, không phải nhân quả. |
| **Đầu ra** | `clean_df` → `q04_leadtime_stats`, `q04_leadtime_cancel` + `fig03_leadtime_cancel.png` |

**Gợi ý:** Lưu **cả 2 bảng**.

In [ ]:
q04 = None   # TODO: bảng q04_leadtime_stats
q04_b = None   # TODO: bảng q04_leadtime_cancel

save_table(q04, 'q04_leadtime_stats')
save_table(q04_b, 'q04_leadtime_cancel')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt phòng; loại … vì …. Nguồn: outputs/tables/q04_leadtime_stats.csv')
# save_fig(fig, 'fig03_leadtime_cancel')
# plt.show()

In [ ]:
check('q04_leadtime_stats đã được tạo', lambda: q04 is not None and len(q04) > 0)
check('q04_leadtime_cancel đã được tạo', lambda: q04_b is not None and len(q04_b) > 0)
check('q04_leadtime_cancel có 5 nhóm', lambda: len(q04_b) == 5)
check('Đã lưu fig03_leadtime_cancel.png', lambda: (OUTPUT_DIR / 'figures' / 'fig03_leadtime_cancel.png').exists())

> **✍️ Nhận xét Câu 4** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt phòng (đã loại … đặt phòng vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig03_leadtime_cancel.png`*

## Câu 5 · Top 10 quốc gia của khách

| | |
|---|---|
| **Cần tính** | Đếm đặt phòng theo `country`; Top 10; tỷ lệ trên **tổng số đặt phòng**; tỷ lệ hủy mỗi nước. Báo cáo riêng số đặt phòng thiếu quốc gia. Đồng hạng → sắp tiếp theo mã. |
| **Mẫu số và trình bày** | Biểu đồ 4: cột ngang, sắp theo số lượng. |
| **Đầu ra** | `clean_df` → `q05_top10_country` + `fig04_top10_country.png` |

**Gợi ý:** Cột gợi ý: `rank, country, n_bookings, pct_of_all_bookings, cancel_rate_pct`.

In [ ]:
q05 = None   # TODO: bảng q05_top10_country

save_table(q05, 'q05_top10_country')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt phòng; loại … vì …. Nguồn: outputs/tables/q05_top10_country.csv')
# save_fig(fig, 'fig04_top10_country')
# plt.show()

In [ ]:
check('q05_top10_country đã được tạo', lambda: q05 is not None and len(q05) > 0)
check('q05 có 10 quốc gia', lambda: len(q05) == 10)
check('Đã lưu fig04_top10_country.png', lambda: (OUTPUT_DIR / 'figures' / 'fig04_top10_country.png').exists())

> **✍️ Nhận xét Câu 5** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt phòng (đã loại … đặt phòng vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig04_top10_country.png`*

## Câu 6 · Phân khúc thị trường

| | |
|---|---|
| **Cần tính** | Theo `market_segment`: số đặt phòng, tỷ trọng, tỷ lệ hủy; báo cáo riêng số thiếu phân khúc. |
| **Mẫu số và trình bày** | Biểu đồ 5: cột ngang (số đặt phòng hoặc tỷ lệ hủy), sắp theo giá trị. Nêu phân khúc nhỏ (ví dụ Aviation) có mẫu ít → tỷ lệ kém ổn định. |
| **Đầu ra** | `clean_df` → `q06_segment_cancel` + `fig05_segment_cancel.png` |

**Gợi ý:** Thêm cột `n_bookings` cạnh tỷ lệ để người đọc thấy cỡ mẫu.

In [ ]:
q06 = None   # TODO: bảng q06_segment_cancel

save_table(q06, 'q06_segment_cancel')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt phòng; loại … vì …. Nguồn: outputs/tables/q06_segment_cancel.csv')
# save_fig(fig, 'fig05_segment_cancel')
# plt.show()

In [ ]:
check('q06_segment_cancel đã được tạo', lambda: q06 is not None and len(q06) > 0)
check('Đã lưu fig05_segment_cancel.png', lambda: (OUTPUT_DIR / 'figures' / 'fig05_segment_cancel.png').exists())

> **✍️ Nhận xét Câu 6** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt phòng (đã loại … đặt phòng vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig05_segment_cancel.png`*

## Câu 7 · Loại đặt cọc và trạng thái đặt phòng

| | |
|---|---|
| **Cần tính** | Bảng chéo `deposit_type × reservation_status`; tỷ lệ **trong từng loại đặt cọc** (Check-Out, Canceled, No-Show, và `Unknown` nếu trạng thái thiếu). |
| **Mẫu số và trình bày** | Biểu đồ 6: cột chồng 100%. So sánh tỷ lệ hủy giữa các loại theo điểm phần trăm và nêu cỡ mẫu (Refundable rất ít). |
| **Đầu ra** | `clean_df` → `q07_deposit_cancel` + `fig06_deposit_cancel.png` |

**Gợi ý:** `pd.crosstab(..., normalize='index')`; điền `'Unknown'` **chỉ khi trình bày**.

In [ ]:
q07 = None   # TODO: bảng q07_deposit_cancel

save_table(q07, 'q07_deposit_cancel')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt phòng; loại … vì …. Nguồn: outputs/tables/q07_deposit_cancel.csv')
# save_fig(fig, 'fig06_deposit_cancel')
# plt.show()

In [ ]:
check('q07_deposit_cancel đã được tạo', lambda: q07 is not None and len(q07) > 0)
check('Đã lưu fig06_deposit_cancel.png', lambda: (OUTPUT_DIR / 'figures' / 'fig06_deposit_cancel.png').exists())

> **✍️ Nhận xét Câu 7** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt phòng (đã loại … đặt phòng vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig06_deposit_cancel.png`*

## Câu 8 · Giá phòng bình quân/đêm (ADR)

| | |
|---|---|
| **Cần tính** | Chỉ đặt phòng **không hủy** có `adr` hợp lệ **và > 0**: theo khách sạn — số mẫu, trung bình, trung vị, min, max, Q1, Q3. Báo cáo riêng số `adr = 0` và số thiếu. |
| **Mẫu số và trình bày** | Biểu đồ 7: histogram độ rộng khoảng 20 EUR, tách hai khách sạn (2 panel hoặc chồng trong suốt). Không xóa giá cao trong miền. |
| **Đầu ra** | `clean_df` → `q08_adr_stats_by_hotel` + `fig07_adr_hist.png` |

**Gợi ý:** `groupby('hotel')['adr'].describe()`.

In [ ]:
q08 = None   # TODO: bảng q08_adr_stats_by_hotel

save_table(q08, 'q08_adr_stats_by_hotel')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt phòng; loại … vì …. Nguồn: outputs/tables/q08_adr_stats_by_hotel.csv')
# save_fig(fig, 'fig07_adr_hist')
# plt.show()

In [ ]:
check('q08_adr_stats_by_hotel đã được tạo', lambda: q08 is not None and len(q08) > 0)
check('Đã lưu fig07_adr_hist.png', lambda: (OUTPUT_DIR / 'figures' / 'fig07_adr_hist.png').exists())

> **✍️ Nhận xét Câu 8** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt phòng (đã loại … đặt phòng vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig07_adr_hist.png`*

## Câu 9 · Thời gian lưu trú

| | |
|---|---|
| **Cần tính** | Đặt phòng **không hủy** có `total_nights` hợp lệ: phân bố theo số đêm (0, 1, …, 14, `15+`) và khách sạn; tỷ lệ lưu trú 1–3 đêm; số đặt phòng 0 đêm (báo cáo riêng). |
| **Mẫu số và trình bày** | Biểu đồ 8: cột theo số đêm **tăng dần**, tách khách sạn. |
| **Đầu ra** | `clean_df` → `q09_stay_nights` + `fig08_stay_nights.png` |

**Gợi ý:** Cột gợi ý: `nights_group, City Hotel, Resort Hotel, total, pct`.

In [ ]:
q09 = None   # TODO: bảng q09_stay_nights

save_table(q09, 'q09_stay_nights')

# TODO: vẽ biểu đồ — mẫu:
# fig, ax = plt.subplots(figsize=FIGSIZE)
# ...
# finish_chart(ax, 'Tiêu đề nói rõ đại lượng, nhóm, giai đoạn', 'Trục x (đơn vị)', 'Trục y (đơn vị)',
#              note='n = … đặt phòng; loại … vì …. Nguồn: outputs/tables/q09_stay_nights.csv')
# save_fig(fig, 'fig08_stay_nights')
# plt.show()

In [ ]:
check('q09_stay_nights đã được tạo', lambda: q09 is not None and len(q09) > 0)
check('Đã lưu fig08_stay_nights.png', lambda: (OUTPUT_DIR / 'figures' / 'fig08_stay_nights.png').exists())

> **✍️ Nhận xét Câu 9** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt phòng (đã loại … đặt phòng vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…` · `outputs/figures/fig08_stay_nights.png`*

## Câu 10 · Đổi loại phòng khi nhận phòng

| | |
|---|---|
| **Cần tính** | Trong đặt phòng **Check-Out** có đủ cả hai loại phòng: tỷ lệ `room_changed_flag` theo khách sạn và theo `reserved_room_type`. |
| **Mẫu số và trình bày** | Mẫu số: số đặt phòng Check-Out có đủ hai loại phòng. Biểu đồ tùy chọn (`fig09_room_change.png`). Không suy ra lý do đổi phòng (nâng hạng hay hết phòng). |
| **Đầu ra** | `clean_df` → `q10_room_change_by_hotel`, `q10_room_change_by_reserved_type` |

**Gợi ý:** Với loại phòng có rất ít đặt phòng, nêu cỡ mẫu bên cạnh tỷ lệ.

In [ ]:
q10 = None   # TODO: bảng q10_room_change_by_hotel
q10_b = None   # TODO: bảng q10_room_change_by_reserved_type

save_table(q10, 'q10_room_change_by_hotel')
save_table(q10_b, 'q10_room_change_by_reserved_type')

In [ ]:
check('q10_room_change_by_hotel đã được tạo', lambda: q10 is not None and len(q10) > 0)
check('q10_room_change_by_reserved_type đã được tạo', lambda: q10_b is not None and len(q10_b) > 0)

> **✍️ Nhận xét Câu 10** *(3 đến 6 câu — viết bằng số liệu thật của nhóm)*
>
> 1. **Phát hiện chính:** …
> 2. **Số liệu và mẫu số:** … trên … đặt phòng (đã loại … đặt phòng vì …)
> 3. **So sánh làm rõ:** … (chênh … điểm phần trăm / gấp … lần)
> 4. **Giới hạn:** dữ liệu không cho biết …
>
> *Nguồn: bảng `outputs/tables/…`*

---
# 6 INSIGHT CÓ BẰNG CHỨNG

Viết **6 insight** từ **ít nhất 4 chủ đề khác nhau** (khách sạn, mùa vụ, lead time, quốc gia, phân khúc, đặt cọc, giá phòng, lưu trú, đổi phòng, chất lượng dữ liệu). Mỗi insight **80–120 từ**, theo `templates/insight_template.md`.

| # | Chủ đề | Câu/Bảng/Hình làm bằng chứng | Phát hiện chính (1 câu) |
|---|---|---|---|
| 1 | … | … | … |
| 2 | … | … | … |
| 3 | … | … | … |
| 4 | … | … | … |
| 5 | … | … | … |
| 6 | … | … | … |

**Tự kiểm tra mỗi insight:** ☐ có số liệu + mẫu số · ☐ có phép so sánh · ☐ nêu ý nghĩa trong phạm vi 2 khách sạn của mẫu · ☐ nêu giới hạn · ☐ trỏ tới bảng/hình · ☐ không dùng từ nhân quả, không suy rộng ra toàn ngành.

**Giới hạn chung (≥ 3) và hướng bổ sung dữ liệu (≥ 2):** …

---
# GIAI ĐOẠN E · XUẤT DỮ LIỆU VÀ TỰ KIỂM TRA TRƯỚC KHI NỘP

1. Chạy ô xuất bên dưới.
2. **Restart & Run all** — mọi ô phải chạy không lỗi, số thứ tự ô liên tục 1, 2, 3…
3. Xem bảng tiến độ: mục tiêu 100%.
4. (Khuyến nghị) Chạy `python tools/kiem_tra_nop_bai.py <thư mục bài nộp>` trên thư mục đã sắp xếp theo Mục 12 của đề.

In [ ]:
outputs = {
    'bookings_clean.csv': clean_df,
    'monthly_summary.csv': monthly_summary,
    'cleaning_log.csv': cleaning_log,
    'row_lineage.csv': row_lineage,
    'quality_report.csv': quality_report,
}
for filename, frame in outputs.items():
    if frame is None:
        print(f'⏳ CHƯA XUẤT {filename}: biến còn None.')
    else:
        frame.to_csv(OUTPUT_DIR / filename, index=False, encoding='utf-8-sig')
        print(f'💾 outputs/{filename}  ({len(frame)} dòng)')

In [ ]:
show_progress()

In [ ]:
# (Tùy chọn — Colab) Nén thư mục outputs để tải về máy trước khi phiên Colab kết thúc
import shutil
archive = shutil.make_archive('outputs_backup', 'zip', OUTPUT_DIR)
print('Đã tạo', archive, '— tải về từ bảng Files.')